<a href="https://colab.research.google.com/github/s-fridman/ab-test-significance-analysis/blob/main/A_B_Test_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Test Analysis: Statistical Significance of Conversion Metrics

[Spreadsheet](https://docs.google.com/spreadsheets/d/1di1E-3kYSSfDCrXkRWYOFNeFSKXnzk8EnXcEZfH6Uac/edit?usp=sharing)  
[Tableau Dashboard](https://public.tableau.com/views/ABTestResults_17946029366250/ABtest?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

## Goal

The goal of this project is to analyze the results of four A/B tests using statistical methods in Python. For each test, I compare the conversion rates of group A (control) and group B (variant) for the key checkout and registration metrics, determine whether the differences are statistically significant, and prepare a results table for further visualization.

Metrics analyzed (conversion = number of events / number of sessions):

- `add_payment_info / session`
- `add_shipping_info / session`
- `begin_checkout / session`
- `new_accounts / session`

Significance is tested with a two-proportion z-test at a significance level of alpha = 0.05. The results are calculated for each test overall and also broken down by country, device, continent and traffic channel.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Mate\ Colab\ stuff

/content/drive/MyDrive/Mate Colab stuff


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

data = pd.read_csv("AB testing tool.csv")
data.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,(not set),desktop,Americas,Direct,2,1,new account,1
1,2020-11-01,(not set),desktop,Asia,Direct,2,1,new account,1
2,2020-11-02,(not set),mobile,Asia,Direct,2,2,new account,1
3,2020-11-02,(not set),desktop,Asia,Organic Search,2,1,new account,1
4,2020-11-02,(not set),mobile,Europe,Organic Search,2,2,new account,1


# Data overview

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB


In [ ]:
data['event_name'].value_counts()

,count
event_name,
session,107210
session_start,106242
page_view,101907
user_engagement,94520
first_visit,81621
scroll,73643
view_promotion,61695
view_item,44869
session_with_orders,25892


In [ ]:
distribution = (data[data['event_name'] == 'session'].pivot_table(index='test', columns='test_group', values='value', aggfunc='sum'))
distribution

test_group,1,2
test,,
1,45362,45193
2,50637,50244
3,70047,70439
4,105079,105141


In [ ]:
data.isna().sum()

,0
date,0
country,0
device,0
continent,0
channel,0
test,0
test_group,0
event_name,0
value,0


In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
print(f"Reviewed period: {data['date'].min()} - {data['date'].max()}")

Reviewed period: 2020-11-01 - 2021-01-27


The dataset contains 800,996 rows and 9 columns, with no missing values and no duplicate rows. The data is aggregated: each row shows how many times a given event occurred for a specific combination of date, country, device, channel, test and test group.
Each test is a separate experiment, so each test is analyzed on its own and the results are not combined.
Test group: 1 = A (control), 2 = B (variant). Test number (1 to 4).
Reviewed period starting from  2020-11-01 to 2021-01-27.

# Data cleaning

In [ ]:
data['date'] = pd.to_datetime(data['date'])
data['date'].dtype

dtype('<M8[ns]')

In [ ]:
data['event_name'] = data['event_name'].replace({'new account': 'new_accounts'})

In [ ]:
data['country'].value_counts()

,count
country,
United States,53507
India,38123
Canada,35165
United Kingdom,26455
(not set),22063
...,...
Bahamas,601
Honduras,593
Bahrain,589


In [ ]:
data['country'] = data['country'].replace({'(not set)': 'Unknown'})
data['country'].value_counts()

,count
country,
United States,53507
India,38123
Canada,35165
United Kingdom,26455
Unknown,22063
...,...
Bahamas,601
Honduras,593
Bahrain,589


In [ ]:
data['continent'].value_counts()

,count
continent,
Europe,289723
Asia,285747
Americas,171834
Africa,28652
Oceania,19040
(not set),6000


In [ ]:
data['continent'] = data['continent'].replace({'(not set)': 'Unknown'})

In [ ]:
data['channel'].value_counts()

,count
channel,
Organic Search,237152
Paid Search,204056
Direct,188716
Social Search,96068
Undefined,75004


In [ ]:
data['channel'] = data['channel'].replace({'Undefined': 'Unknown'})

Unifying unknown values for easier visualization.

In [ ]:
data['test'].value_counts()

,count
test,
4,302824
3,200757
2,156791
1,140624


Unknown values ('(not set)' and 'Undefined') were unified as 'Unknown', and the event 'new account' was renamed to 'new_accounts' to match the metric name. The event name new account contains a space and differs from the metric name new_accounts. I renamed it to new_accounts so that all metrics can be stored in a single list and processed in a loop without a separate name mapping.

# Statistical significance calculation

In [ ]:
alpha = 0.05

metrics = ['add_payment_info',
           'add_shipping_info',
           'begin_checkout',
           'new_accounts']

The significance level is set to alpha = 0.05. The metrics are stored in a list, so a new metric can be added with a single line, and all calculations run in loops over this list.

In [ ]:
columns = ['test', 'country', 'device', 'continent', 'channel']
needed = metrics + ['session']

agg = data[data['event_name'].isin(needed)].groupby(columns + ['test_group', 'event_name'],
                                            as_index=False)['value'].sum().pivot_table(index=columns + ['test_group'],
                                            columns='event_name', values='value', fill_value=0).reset_index()
agg[needed] = agg[needed].astype(int)

agg.head()

event_name,test,country,device,continent,channel,test_group,add_payment_info,add_shipping_info,begin_checkout,new_accounts,session
0,1,Albania,desktop,Europe,Direct,1,0,0,0,0,1
1,1,Albania,desktop,Europe,Organic Search,1,0,0,0,0,1
2,1,Albania,desktop,Europe,Organic Search,2,0,0,0,1,5
3,1,Albania,desktop,Europe,Paid Search,1,1,1,2,0,1
4,1,Albania,desktop,Europe,Paid Search,2,0,0,0,0,4


The data is aggregated to one row per combination of test, country, device, continent, channel and test group. Event counts (the sum of 'value') are placed in separate columns, one per metric plus 'session'. This makes it easy to sum events and sessions for any slice of the data.

In [ ]:
rows = []

for test in agg['test'].unique():
    df = agg[agg['test'] == test]
    a = df[df['test_group'] == 1]
    b = df[df['test_group'] == 2]

    sessions_a = a['session'].sum()
    sessions_b = b['session'].sum()

    for metric in metrics:
        events_a = a[metric].sum()
        events_b = b[metric].sum()

        conv_a = events_a / sessions_a
        conv_b = events_b / sessions_b

        z_stat, p_value = proportions_ztest([events_b, events_a], [sessions_b, sessions_a])

        rows.append({
            'test_number': test,
            'breakdown': 'total',
            'segment': 'all',
            'metric': metric,
            'numerator_event': metric,
            'denominator_event': 'session',
            'numerator_count_A': events_a,
            'denominator_count_A': sessions_a,
            'conversion_rate_A': conv_a,
            'numerator_count_B': events_b,
            'denominator_count_B': sessions_b,
            'conversion_rate_B': conv_b,
            'metric_change': (conv_b - conv_a) / conv_a * 100 if conv_a > 0 else np.nan,
            'z_stat': z_stat,
            'p_value': p_value,
            'significant': p_value < alpha})

results_total = pd.DataFrame(rows)
results_total

,test_number,breakdown,segment,metric,numerator_event,denominator_event,numerator_count_A,denominator_count_A,conversion_rate_A,numerator_count_B,denominator_count_B,conversion_rate_B,metric_change,z_stat,p_value,significant
0,1,total,all,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,1,total,all,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
2,1,total,all,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
3,1,total,all,new_accounts,new_accounts,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
4,2,total,all,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
5,2,total,all,add_shipping_info,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
6,2,total,all,begin_checkout,begin_checkout,session,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False
7,2,total,all,new_accounts,new_accounts,session,4165,50637,0.082252,4184,50244,0.083274,1.241934,0.588793,0.556000,False
8,3,total,all,add_payment_info,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,1.474630,0.643172,0.520112,False
9,3,total,all,add_shipping_info,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,-2.621211,-1.413727,0.157442,False


For each test and each metric, I calculated:

- the number of events and sessions in groups A and B;
- the conversion rate in each group;
- 'metric_change', the relative change of B compared to A, in percent;
- the z-statistic and p-value of a two-proportion z-test;
- 'significant', which is True if p-value < alpha.

The tests are independent experiments, so each test is analyzed separately and the results are not combined.

In [ ]:
rows_breakdown = []
checked = 0
no_sessions = 0
no_events = 0
skipped = 0

for c in ['country', 'device', 'continent', 'channel']:
    for value in agg[c].unique():
        for test in agg['test'].unique():
            df = agg[(agg['test'] == test) & (agg[c] == value)]
            a = df[df['test_group'] == 1]
            b = df[df['test_group'] == 2]

            sessions_a = a['session'].sum()
            sessions_b = b['session'].sum()

            if sessions_a == 0 or sessions_b == 0:
                no_sessions += 1
                continue

            for metric in metrics:
                checked += 1
                events_a = a[metric].sum()
                events_b = b[metric].sum()

                if events_a + events_b == 0:
                    no_events += 1
                    continue

                if events_a >= sessions_a or events_b >= sessions_b:
                    skipped += 1
                    continue

                conv_a = events_a / sessions_a
                conv_b = events_b / sessions_b
                z_stat, p_value = proportions_ztest([events_b, events_a], [sessions_b, sessions_a])

                rows_breakdown.append({
                    'test_number': test,
                    'breakdown': c,
                    'segment': value,
                    'metric': metric,
                    'numerator_event': metric,
                    'denominator_event': 'session',
                    'numerator_count_A': events_a,
                    'denominator_count_A': sessions_a,
                    'conversion_rate_A': conv_a,
                    'numerator_count_B': events_b,
                    'denominator_count_B': sessions_b,
                    'conversion_rate_B': conv_b,
                    'metric_change': (conv_b - conv_a) / conv_a * 100 if conv_a > 0 else np.nan,
                    'z_stat': z_stat,
                    'p_value': p_value,
                    'significant': p_value < alpha})

results_breakdown = pd.DataFrame(rows_breakdown)

print("Segments checked:", checked)
print("Segments calculated:", len(results_breakdown))
print("Skipped, no sessions in a group:", no_sessions)
print("Skipped, no events:", no_events)
print("Skipped, events >= sessions:", skipped)
results_breakdown.head()

Segments checked: 1952
Segments calculated: 1793
Skipped, no sessions in a group: 0
Skipped, no events: 157
Skipped, events >= sessions: 2


,test_number,breakdown,segment,metric,numerator_event,denominator_event,numerator_count_A,denominator_count_A,conversion_rate_A,numerator_count_B,denominator_count_B,conversion_rate_B,metric_change,z_stat,p_value,significant
0,1,country,Albania,add_payment_info,add_payment_info,session,1,9,0.111111,0,16,0.000000,-100.0,-1.360828,0.173568,False
1,1,country,Albania,add_shipping_info,add_shipping_info,session,2,9,0.222222,0,16,0.000000,-100.0,-1.965893,0.049311,True
2,1,country,Albania,begin_checkout,begin_checkout,session,3,9,0.333333,0,16,0.000000,-100.0,-2.461830,0.013823,True
3,1,country,Albania,new_accounts,new_accounts,session,0,9,0.000000,1,16,0.062500,NaN,0.765466,0.443994,False
4,2,country,Albania,add_payment_info,add_payment_info,session,0,14,0.000000,1,14,0.071429,NaN,1.018350,0.308512,False


### Breakdowns

The same calculation is repeated for each test within every country, device, continent and channel. Segments that can not be tested are skipped:

- segments with no events in both groups (157 cases): conversion is 0 in both groups, so there is nothing to compare;
- segments where the number of events is not smaller than the number of sessions (2 cases): conversion is not a proportion in this case, so the z-test does not apply.

In total, 1,793 of 1,952 test-metric-segment combinations were calculated.

Very small segments (only a few sessions) can show a "significant" result by chance, so the number of sessions is saved in 'denominator_count_A' and 'denominator_count_B' and can be used for filtering.

In [ ]:
final_results = pd.concat([results_total, results_breakdown], ignore_index=True)
final_results.to_csv("ab_test_results.csv", index=False)
final_results.head()

,test_number,breakdown,segment,metric,numerator_event,denominator_event,numerator_count_A,denominator_count_A,conversion_rate_A,numerator_count_B,denominator_count_B,conversion_rate_B,metric_change,z_stat,p_value,significant
0,1,total,all,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,1,total,all,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
2,1,total,all,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
3,1,total,all,new_accounts,new_accounts,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
4,2,total,all,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False


In [ ]:
bd = final_results[final_results['breakdown'] != 'total']
print(f"{bd['significant'].sum()}, {round(bd['significant'].mean() * 100, 1)}%")

summary = bd.groupby('breakdown')['significant'].agg(['sum', 'count', 'mean'])
summary['share, %'] = (summary['mean'] * 100).round(1)
print(summary[['sum', 'count', 'share, %']])

369, 20.6%
           sum  count  share, %
breakdown                      
channel     32     80      40.0
continent   28     96      29.2
country    292   1569      18.6
device      17     48      35.4


# Conclusions

For each metric, the null hypothesis states that the conversion rates in groups A and B are equal. H0 is rejected when the p-value is below alpha = 0.05. The direction of the difference is given by 'metric_change' (positive means that the conversion in group B is higher than in group A).

**Test 1.** Three of four metrics show a statistically significant difference, and in all three the conversion in group B is higher:
- add_payment_info: p < 0.001, +12.5%
- add_shipping_info: p = 0.009, +6.6%
- begin_checkout: p = 0.003, +6.7%

'new_accounts' is not significant (p = 0.123, -3.4%), so H0 can not be rejected for this metric.

**Test 2.** No metric shows a significant difference (p-values from 0.21 to 0.56), so H0 can not be rejected for any of the four metrics.

**Test 3.** Only 'begin_checkout' is significant (p = 0.012), and the conversion in group B is lower (-3.4%). The other three metrics are not significant (p-values from 0.157 to 0.520).

**Test 4.** Two metrics are significant, and in both the conversion in group B is lower:
- begin_checkout: p = 0.046, -2.4%
- new_accounts: p = 0.018, -3.4%

In total, 6 of 16 test-metric combinations are significant at alpha = 0.05: 3 in Test 1, 1 in Test 3 and 2 in Test 4.

The p-value is the probability of getting a difference like this one, or larger, if H0 is true. The size of the effect is described by 'metric_change'.

Breakdowns include 1,793 test-metric-segment comparisons. 369 of them (20.6%) are significant at alpha = 0.05. If there were no real differences, only about 5% would be significant by chance, so the result is much higher than expected.

The share of significant results is higher for device (35%), channel (40%) and continent (29%) than for country (19%). These breakdowns have fewer segments with more sessions in each, so the test can find smaller differences more easily. Segments with very few sessions give unreliable p-values, so 'denominator_count_A' and 'denominator_count_B' should be checked when reading the results.
The breakdown comparisons are not independent: the same sessions appear in several breakdowns and in the overall test. Because of this, these shares are descriptive only.